## Bonus Code for Chapter 5

### Alternative Weight Loading from Hugging Face Model Hub Via `safetensors`

* In the main chapter, we loaded the GPT model weights directly from OpenAI
* This notebook provides alternative weight loading code to model weights from the Hugging Face Model Hub using .`safetensors` file

* This is conceptually the same as loading weights of a PyTorch model from via the state-dict method described in chapter 5:

* The appeal of `.safetensors` files lies in their secure design, as they only store tensor data and avoid the execution of potentially malicious code during loading

* In newer versions of PyTorch (e.g., 2.0 and newer), a `weights_only=True` argument can be used with `torch.load` (e.g., `torch.load("model_state_dict.pth", weights_only=True)`) to improve safety by skipping the execution of code and loading only the weights (this is now enabled by default in PyTorch 2.6 and newer)

In [1]:
!pip show safetensors

Name: safetensors
Version: 0.7.0
Summary: 
Home-page: https://github.com/huggingface/safetensors
Author: 
Author-email: Nicolas Patry <patry.nicolas@protonmail.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: accelerate, diffusers, peft, timm, torchtune, transformers


In [2]:
from importlib.metadata import version

pkgs = ["numpy", "torch", "safetensors"]
for p in pkgs:
  print(f"{p} version:{version(p)}")

numpy version:2.0.2
torch version:2.9.0+cpu
safetensors version:0.7.0


In [3]:
# from llms_from_scratch.ch04 import GPTModel
# I use the GPTModel.py file as I am using the colab, one can change and uncomment the above code
# as the file is given in the same folder it is the best approach
from GPTModel import GPTModel



In [5]:
BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},

}

CHOOSE_MODEL = "gpt2-small (124M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [6]:
import os
import requests

from safetensors.torch import load_file

URL_DIR = {
    "gpt2-small (124M)": "gpt2",
    "gpt2-medium (355M)": "gpt2-medium",
    "gpt2-large (774M)": "gpt2-large",
    "gpt2-xl (1558M)": "gpt2-xl"
}


url = f"https://huggingface.co/openai-community/{URL_DIR[CHOOSE_MODEL]}/resolve/main/model.safetensors"
output_file = f"model-{URL_DIR[CHOOSE_MODEL]}.safetensors"

# Download the file
if not os.path.exists(output_file):
  response = requests.get(url, timeout=30)
  response.raise_for_status()
  with open(output_file, "wb") as f:
    f.write(response.content)
# Load the file

state_dict = load_file(output_file)


In [10]:
import torch
import torch.nn as nn

def assign(left, right):
  if left.shape != right.shape:
    raise ValueError(f"Shape mismatch. Left : {left.shape}, Right: {right.shape}")
  return torch.nn.Parameter(right.detach())

In [13]:
import torch


def load_weights_into_gpt(gpt, params):
    """
    Load GPT-2 style pretrained weights into a custom GPT model.
    """

    # ------------------------------------------------------------
    # Embeddings
    # ------------------------------------------------------------
    gpt.pos_emb.weight = assign(
        gpt.pos_emb.weight, params["wpe.weight"]
    )
    gpt.tok_emb.weight = assign(
        gpt.tok_emb.weight, params["wte.weight"]
    )

    # ------------------------------------------------------------
    # Transformer blocks
    # ------------------------------------------------------------
    for b in range(len(gpt.trf_blocks)):

        # -----------------------------
        # Attention: QKV
        # -----------------------------
        q_w, k_w, v_w = torch.chunk(
            params[f"h.{b}.attn.c_attn.weight"], 3, dim=1
        )

        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T
        )
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T
        )
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T
        )

        q_b, k_b, v_b = torch.chunk(
            params[f"h.{b}.attn.c_attn.bias"], 3, dim=0
        )

        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b
        )
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b
        )
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b
        )

        # -----------------------------
        # Attention output projection
        # -----------------------------
        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params[f"h.{b}.attn.c_proj.weight"].T
        )
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params[f"h.{b}.attn.c_proj.bias"]
        )

        # -----------------------------
        # Feed-Forward Network (MLP)
        # -----------------------------
        # c_fc
        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params[f"h.{b}.mlp.c_fc.weight"].T
        )
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params[f"h.{b}.mlp.c_fc.bias"]
        )

        # c_proj
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params[f"h.{b}.mlp.c_proj.weight"].T
        )
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params[f"h.{b}.mlp.c_proj.bias"]
        )

        # -----------------------------
        # LayerNorms
        # -----------------------------
        # ln_1
        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params[f"h.{b}.ln_1.weight"]
        )
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params[f"h.{b}.ln_1.bias"]
        )

        # ln_2
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params[f"h.{b}.ln_2.weight"]
        )
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params[f"h.{b}.ln_2.bias"]
        )

    # ------------------------------------------------------------
    # Final LayerNorm
    # ------------------------------------------------------------
    gpt.final_norm.scale = assign(
        gpt.final_norm.scale, params["ln_f.weight"]
    )
    gpt.final_norm.shift = assign(
        gpt.final_norm.shift, params["ln_f.bias"]
    )

    # ------------------------------------------------------------
    # Language model head
    # ------------------------------------------------------------
    # ⚠ Only use this if lm_head is NOT weight-tied with tok_emb
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte.weight"])

In [14]:
import torch

gpt = GPTModel(BASE_CONFIG)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
load_weights_into_gpt(gpt, state_dict)

gpt.to(device)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_f

In [16]:
import tiktoken
# from llms_from_scratch.ch05 import generate, text_to_token_ids, token_ids_to_text
from gpt_generate import generate, text_to_token_ids, token_ids_to_text


torch.manual_seed(123)

tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate(
    model=gpt.to(device),
    idx=text_to_token_ids("The meaning of life is", tokenizer),
    max_new_tokens=100,
    context_size=BASE_CONFIG["context_length"],
    top_k=1,
    temperature=1.0
)

print("Output text:\n". token_ids_to_text(token_ids, tokenizer))

ModuleNotFoundError: No module named 'previous_chapters'